# Negative Controls - Direct Arms (T21 RAG, GPT, Claude, Nemotron)

Runs the 30 negative-control questions through the FOUR direct benchmark arms
only, with per-question checkpointing. The other two arms and the analysis
live in their own notebooks:

- MedRAG: notebook 13 (medrag kernel)
- PaperQA2: notebook 14 (paperqa kernel)
- Merge + refusal classification + Excel + figure: notebook 15 (dschat kernel)

A negative control is a question the system SHOULD refuse: an off-topic
domain, a fictional or non-existent biological entity, or a biomedical
question with no Down syndrome connection.

Generation mirrors notebook 3's callers exactly (same prompts, retries and
T21 Summary extraction); only the question subset changes (CTRL_Type ==
Negative). Resume-safe: answers are checkpointed per question to
direct_answers_negative.json, and a stored answer is reused unless it is
empty or carries an API-error marker (error strings are never resumed as
answers). Run with the dschat kernel.

## Cell 1 - Configuration

In [ ]:
# ---- Cell 1: configuration ----
import json, os, re, sys, time, random
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

NB_DIR   = Path.cwd()
ANALYSIS = NB_DIR.parent if NB_DIR.name == 'nbs' else NB_DIR
load_dotenv(ANALYSIS / '.env')

EXCEL_PATH = ANALYSIS / 'input' / 'Revised_DownSyndrome_60_Questions_Final_linked.xlsx'
OUT_DIR    = ANALYSIS / 'output' / 'ScientificReportRevisionV1' / 'negative_control_refusals'
OUT_DIR.mkdir(parents=True, exist_ok=True)

T21_API_URL = os.getenv('T21_API_URL', 'http://localhost:8698/answer')

OPENAI_KEY      = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_KEY   = os.getenv('ANTHROPIC_API_KEY', '')
OPENAI_MODEL    = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.4')
ANTHROPIC_MODEL = os.getenv('ANTHROPIC_CHAT_MODEL', 'claude-sonnet-4-6')

OLLAMA_API_KEY        = os.getenv('OLLAMA_API_KEY', '')
OLLAMA_NEMOTRON_MODEL = os.getenv('OLLAMA_NEMOTRON_MODEL', 'nemotron-3-super')
OLLAMA_BASE_URL       = os.getenv('OLLAMA_REMOTE_BASE_URL', 'https://ollama.com')

TOP_K     = 20     # base retrieval count for the T21 arm (dynamic boost applied)
MAX_WORDS = 150
API_SLEEP = 1.5    # seconds between T21 /answer calls

assert OPENAI_KEY,     'OPENAI_API_KEY not set in .env'
assert ANTHROPIC_KEY,  'ANTHROPIC_API_KEY not set in .env'
assert OLLAMA_API_KEY, 'OLLAMA_API_KEY not set in .env'

# Live collection gate (fresh-results round 2026-09-06): the T21 arm must run
# against the same corpus every other benchmark notebook used.
EXPECTED_COLLECTION = 'DS_INCLUDE_2026_09_02_DEEPINFRA_2048'
EXPECTED_VECTORS, EXPECTED_DOCS = 65_677, 1_594
_hj = requests.get(T21_API_URL.replace('/answer', '/health'), timeout=30).json()
_hc = _hj['config']['milvus_collection']
_hv = int(str(_hj['database']['total_vectors']).replace(',', ''))
_hd = int(str(_hj['database']['estimated_documents']).replace(',', ''))
assert (_hc, _hv, _hd) == (EXPECTED_COLLECTION, EXPECTED_VECTORS, EXPECTED_DOCS), (
    f'API serves {_hc} ({_hv:,}/{_hd:,}); expected {EXPECTED_COLLECTION} '
    f'({EXPECTED_VECTORS:,}/{EXPECTED_DOCS:,})')
print(f'Collection gate OK: {_hc} ({_hv:,} vectors, {_hd:,} documents)')
print('Questions  :', EXCEL_PATH.name)
print('Output dir :', OUT_DIR)
print('Arms       :', f'T21 RAG | GPT ({OPENAI_MODEL}) | Claude ({ANTHROPIC_MODEL}) | '
      f'Nemotron ({OLLAMA_NEMOTRON_MODEL} @ {OLLAMA_BASE_URL})')


## Cell 2 - The 30 negative-control questions

In [ ]:
# ---- Cell 2: the 30 negative-control questions ----
df_q = pd.read_excel(EXCEL_PATH, sheet_name='Questions_for_testing')
df_q.columns = [c.strip() for c in df_q.columns]
neg = df_q[df_q['CTRL_Type'].str.strip().str.lower() == 'negative'].copy().reset_index(drop=True)
neg = neg.rename(columns={'Original question': 'question'})
neg['question'] = neg['question'].astype(str).str.strip()
neg['sl_no'] = neg['SLNo'].astype(str).str.strip().str.replace(r'^[Qq]', '', regex=True)
assert len(neg) == 30, f'expected 30 negative controls, found {len(neg)}'
print(f'negative controls: {len(neg)}')
neg[['sl_no', 'question']]


## Cell 3 - Helpers and model callers

Identical implementations to notebook 3 Cell 5 (clean-up, T21 Summary
extraction, dynamic top_k, prose system prompt, retry logic). The three chat
arms receive exactly the same prompt they received on the positive controls,
including the closing scope instruction: "If the question is outside the scope
of Down syndrome / Trisomy 21 research, state that clearly in one sentence."
No ground-truth length adaptation is applied here (negative controls have no
ground truth), so the defaults of 150 words / 5 sentences are used.
call_claude additionally guards against an empty content list (observed once
on a negative control: r.content[0] raised IndexError) by treating it as a
transient error and retrying.

In [ ]:
# ---- Cell 3: helpers and callers (mirrors notebook 3) ----
from openai import OpenAI
from anthropic import Anthropic
from ollama import Client as OllamaClient

openai_client    = OpenAI(api_key=OPENAI_KEY)
anthropic_client = Anthropic(api_key=ANTHROPIC_KEY)
ollama_client    = OllamaClient(
    host=OLLAMA_BASE_URL,
    headers={'Authorization': f'Bearer {OLLAMA_API_KEY}'},
)

_SPACE_BEFORE_PUNCT = re.compile(r'\s+([.,;:!?])')
_ORPHAN_MARKER_RE   = re.compile(r'(?:(?<=^)|(?<=\s))(?:[1-9]|1[0-2])(?=[.,;:!?](?:\s|$))')

def clean_response(text: str) -> str:
    text = str(text or '')
    text = _ORPHAN_MARKER_RE.sub('', text)
    text = _SPACE_BEFORE_PUNCT.sub(r'\1', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

def strip_quality_review(md: str) -> str:
    s = re.sub(r'(?ims)^\s*#{1,6}\s*Quality\s+Review\s*\n.*?(?=^\s*#{1,6}\s*\S|\Z)', '', md)
    s = re.sub(r'(?ims)^\s*Quality\s+Review\s*:?\s*\n.*?(?=^\s*#{1,6}\s*\S|\Z)', '', s)
    return re.sub(r'\n{3,}', '\n\n', s).strip()

def extract_t21_answer(payload: dict) -> str:
    '''Extract the # Summary section, stripped of all citation markers.'''
    md = str(payload.get('markdown') or '')
    if md.strip():
        md = strip_quality_review(md)
        m = re.search(r'#{1,2}\s*Summary[^\n]*\n(.*?)(?=\n#{1,2}\s|\Z)', md,
                      flags=re.DOTALL | re.IGNORECASE)
        txt = (m.group(1).strip() if m else md).strip()
        txt = re.sub(r'\s*\[{1,2}PMC\d{4,}:[A-Za-z0-9_-]+\]{1,2}(?:\([^)]+\))?', '', txt)
        txt = re.sub(r'\[{1,2}PMC\d{4,}\]{1,2}(?:\([^)]+\))?', '', txt)
        return clean_response(txt)
    for k in ('answer', 'response', 'text', 'result'):
        v = str(payload.get(k) or '').strip()
        if v:
            return clean_response(v)
    return ''

_COMPLEX_KEYWORDS = [
    'compare', 'versus', 'vs', 'difference', 'similar', 'contrast',
    'all', 'comprehensive', 'complete', 'overview', 'summarize',
    'how many', 'list', 'what are the', 'types of', 'categories',
    'mechanisms', 'pathways', 'treatments', 'therapies',
    'subgroups', 'subtypes', 'classification',
    'which', 'what conditions', 'what factors', 'what genes',
    'lower risk', 'higher risk', 'protective', 'risk factors',
    'associated with', 'related to', 'linked to',
]

def _get_dynamic_top_k(question: str, base_k: int = TOP_K, max_k: int = 30) -> int:
    q_lower = question.lower()
    boost = 0
    for kw in _COMPLEX_KEYWORDS:
        if kw in q_lower:
            boost = 5
            break
    if len(question) > 100:
        boost = max(boost, 3)
    if ' and ' in q_lower and len(question) > 50:
        boost = max(boost, 4)
    return min(base_k + boost, max_k)

def _make_prose_system(max_words: int = MAX_WORDS, max_sents: int = 5) -> str:
    return (
        'You are a biomedical research assistant specializing in Down syndrome (trisomy 21) research. '
        f'Write your answer as a continuous prose paragraph of exactly {max_sents} complete sentence(s) '
        f'and approximately {max_words} words. '
        'Do NOT use bullet points, numbered lists, bold headers, or markdown formatting. '
        'Do NOT include citations, references, or source markers of any kind. '
        'Do NOT open with a single-word answer such as "Yes." or "No." - begin directly with substantive information. '
        'Be factual, specific, and include quantitative data (percentages, effect sizes, sample sizes) where available. '
        'If the question is outside the scope of Down syndrome / Trisomy 21 research, '
        'state that clearly in one sentence.'
    )

def _is_transient_t21_error(err_text: str) -> bool:
    t = str(err_text or '').lower()
    return ('timed out' in t or 'timeout' in t or 'connection' in t
            or 'read timeout' in t or '502' in t or '503' in t or '504' in t)

def call_t21(question: str, max_attempts: int = 4) -> tuple[str, dict]:
    top_k = _get_dynamic_top_k(question)
    for attempt in range(1, max_attempts + 1):
        timeout_s = 900 + (attempt - 1) * 60
        try:
            r = requests.post(T21_API_URL,
                              json={'question': question, 'top_k': top_k, 'include_hits': False},
                              timeout=timeout_s)
            r.raise_for_status()
            payload = r.json()
            return extract_t21_answer(payload), payload
        except requests.HTTPError as e:
            status = e.response.status_code if e.response is not None else 'NA'
            print(f'  [T21_WARN] HTTP {status}')
            return '', {'error': str(e)}
        except Exception as e:
            err = str(e)
            if attempt < max_attempts and _is_transient_t21_error(err):
                sleep_s = 2 ** attempt + random.uniform(0, 1)
                print(f'  [T21_WARN] attempt {attempt}/{max_attempts}; retrying in {sleep_s:.0f}s')
                time.sleep(sleep_s)
                continue
            print(f'  [T21_FAIL] {err[:120]}')
            return '', {'error': err}
    return '', {'error': 'max retries exceeded'}

def call_openai(question: str) -> str:
    try:
        r = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{'role': 'system', 'content': _make_prose_system()},
                      {'role': 'user',   'content': question}],
            temperature=0.2)
        return (r.choices[0].message.content or '').strip()
    except Exception as e:
        return f'[OPENAI_ERROR] {e}'

def _is_transient_claude_error(err_text: str) -> bool:
    t = str(err_text or '').lower()
    return ('error code: 500' in t or 'internal server error' in t
            or "'type': 'api_error'" in t or 'overloaded' in t
            or 'rate limit' in t or '529' in t)

def call_claude(question: str, max_attempts: int = 5) -> str:
    for attempt in range(1, max_attempts + 1):
        try:
            r = anthropic_client.messages.create(
                model=ANTHROPIC_MODEL, max_tokens=400, temperature=0.2,
                system=_make_prose_system(),
                messages=[{'role': 'user', 'content': question}])
            # API-level refusal: the model declines with stop_reason='refusal'
            # and an empty content list (observed on negative control SLNo 50).
            # That IS a refusal - record it as one, not as an error.
            if getattr(r, 'stop_reason', '') == 'refusal':
                return ('[NO RESPONSE] The model declined to answer '
                        '(Anthropic API stop_reason=refusal, no content returned).')
            # Guard: r.content can otherwise be an empty list -
            # treat an empty response as transient and retry.
            txt = ' '.join(getattr(b, 'text', '') for b in (r.content or [])).strip()
            if txt:
                return txt
            if attempt < max_attempts:
                print(f'  [CLAUDE_WARN] empty response, attempt {attempt}/{max_attempts}; retrying')
                time.sleep(2 ** attempt + random.uniform(0, 1))
                continue
            return '[CLAUDE_ERROR] empty response after retries'
        except Exception as e:
            err = str(e)
            if attempt < max_attempts and _is_transient_claude_error(err):
                sleep_s = 2 ** attempt + random.uniform(0, 1)
                print(f'  [CLAUDE_WARN] attempt {attempt}/{max_attempts}; retrying in {sleep_s:.1f}s')
                time.sleep(sleep_s)
                continue
            return f'[CLAUDE_ERROR] {err[:160]}'
    return '[CLAUDE_ERROR] max retries exceeded'

def call_nemotron(question: str, max_attempts: int = 6) -> str:
    for attempt in range(1, max_attempts + 1):
        try:
            full_text = ''
            for part in ollama_client.chat(
                    model=OLLAMA_NEMOTRON_MODEL,
                    messages=[{'role': 'system', 'content': _make_prose_system()},
                              {'role': 'user',   'content': question}],
                    stream=True,
                    options={'temperature': 0.2}):
                full_text += (part.message.content or '')
            return clean_response(full_text)
        except Exception as e:
            err = str(e)
            if attempt < max_attempts:
                if '429' in err or 'too many concurrent' in err.lower():
                    sleep_s = min(120, 15 * attempt) + random.uniform(0, 3)
                else:
                    sleep_s = 2 ** attempt + random.uniform(0, 1)
                print(f'  [NEMOTRON_WARN] attempt {attempt}/{max_attempts}; '
                      f'retrying in {sleep_s:.1f}s - {err[:80]}')
                time.sleep(sleep_s)
                continue
            return f'[NEMOTRON_ERROR] {err[:160]}'
    return '[NEMOTRON_ERROR] max retries exceeded'

_ERR_RE = re.compile(r'\[[A-Z][A-Z0-9_]*_ERROR\]|too many concurrent', re.I)
print('Helpers OK')


## Cell 4 - Generate the four direct arms

Checkpointed per question to direct_answers_negative.json and resume-safe: a
stored answer is reused unless it is empty or carries an API-error marker.

In [ ]:
# ---- Cell 4: generate direct-arm answers ----
ANS_DIRECT = OUT_DIR / 'direct_answers_negative.json'
direct_answers = json.loads(ANS_DIRECT.read_text()) if ANS_DIRECT.exists() else {}
_DIRECT_KEYS = ('t21', 'gpt', 'claude', 'nemotron')

for i, row in neg.iterrows():
    key, q = row['sl_no'], row['question']
    rec = direct_answers.get(key, {'question': q})
    todo = [k for k in _DIRECT_KEYS
            if not rec.get(f'{k}_answer') or _ERR_RE.search(rec.get(f'{k}_answer', ''))]
    if not todo:
        print(f'  [{i+1}/30] [RESUME] SLNo {key}')
        continue
    print(f'  [{i+1}/30] SLNo {key} ({"+".join(todo)}) | {q[:70]}')
    t0 = time.time()
    if 't21' in todo:
        txt, payload = call_t21(q)
        rec['t21_answer']   = txt
        rec['t21_markdown'] = str(payload.get('markdown') or '')
        time.sleep(API_SLEEP)
    if 'gpt' in todo:
        rec['gpt_answer'] = call_openai(q)
    if 'claude' in todo:
        rec['claude_answer'] = call_claude(q)
    if 'nemotron' in todo:
        rec['nemotron_answer'] = call_nemotron(q)
    rec['secs'] = round(time.time() - t0, 1)
    direct_answers[key] = rec
    ANS_DIRECT.write_text(json.dumps(direct_answers, indent=1, ensure_ascii=False))

for k in _DIRECT_KEYS:
    neg[f'{k}_answer'] = [direct_answers.get(s, {}).get(f'{k}_answer', '') for s in neg['sl_no']]

for k in _DIRECT_KEYS:
    n_ok  = int(sum(1 for v in neg[f'{k}_answer'] if v and not _ERR_RE.search(v)))
    n_err = int(sum(1 for v in neg[f'{k}_answer'] if v and _ERR_RE.search(v)))
    print(f'{k:9s} answered: {n_ok}/30' + (f'  (error markers: {n_err} - re-run this cell to retry)' if n_err else ''))


---

Output: direct_answers_negative.json in
analysis/output/ScientificReportRevisionV1/negative_control_refusals/.

Next steps:
1. Notebook 13 (medrag kernel) - MedRAG answers
2. Notebook 14 (paperqa kernel) - PaperQA2 answers
3. Notebook 15 (dschat kernel) - merge all six arms, classify refusals, save
   the Excel workbook and the figure